# Featuresmith Tutorial: 01 — Getting Started with Featuresmith v0.2.0

Learn the fundamentals of Featuresmith — loading tabular datasets, deterministic statistical profiling, automated dataset code reviews, and ML readiness scoring.

---


## 1. Problem Statement & Why This Capability Matters
Machine learning failure modes often originate from dataset quality rather than model architecture choices. Featuresmith brings developer-first code review discipline to tabular datasets before training begins.

### Objectives
1. Understand Featuresmith's package structure (`featuresmith-core` and `featuresmith-cli`).
2. Load tabular data from CSV, Parquet, Excel, or DataFrames using `fs.load()`.
3. Profile datasets deterministically with `fs.profile()`.
4. Perform an automated dataset code review with `fs.review()`.
5. Extract an explainable 0–100 ML Readiness Score with `fs.score()`.

### Step 1: Import Featuresmith & Verify Version

In [1]:
import os

import featuresmith as fs

print(f"Featuresmith Version: {fs.__version__}")

Featuresmith Version: 0.2.0


### Step 2: Load the Titanic Dataset

In [2]:
data_path = os.path.join("..", "data", "processed", "titanic.csv")
dataset = fs.load(data_path)

print(f"Dataset Source : {dataset.source}")
print(f"Row Count      : {dataset.row_count}")
print(f"Column Count   : {dataset.column_count}")
print(f"Columns        : {dataset.schema.names}")

Dataset Source : ..\data\processed\titanic.csv
Row Count      : 891
Column Count   : 12
Columns        : ('passengerid', 'survived', 'pclass', 'name', 'sex', 'age', 'sibsp', 'parch', 'ticket', 'fare', 'cabin', 'embarked')


### Step 3: Run Vectorized Deterministic Profiling

In [3]:
profile = fs.profile(dataset)
print(f"Overall Missingness: {profile.dataset_summary.missing_percentage:.2f}%")
print("\nSample Column Profiles:")
for col, col_prof in list(profile.column_profiles.items())[:5]:
    print(
        f"  - {col:<15}: logical_type={col_prof.logical_type:<12} missing={col_prof.missing_count}"
    )

Overall Missingness: 8.10%

Sample Column Profiles:
  - passengerid    : logical_type=numeric      missing=0
  - survived       : logical_type=numeric      missing=0
  - pclass         : logical_type=numeric      missing=0
  - name           : logical_type=text         missing=0
  - sex            : logical_type=categorical  missing=0


### Step 4: Run Automated Dataset Review & Scorecard

In [4]:
review_result = fs.review(dataset, target_column="survived")
scorecard = fs.score(review_result)

if scorecard:
    print(f"ML Readiness Score: {scorecard.overall:.1f} / 100")
    print("\nDimension Breakdown:")
    for dim in scorecard.dimensions:
        print(
            f"  - {dim.label:<20}: {dim.score:5.1f}/100 ({len(dim.contributing_findings)} findings)"
        )

ML Readiness Score: 86.9 / 100

Dimension Breakdown:
  - Schema Health       : 100.0/100 (0 findings)
  - Missing Values      :  70.0/100 (1 findings)
  - Duplicate Records   : 100.0/100 (0 findings)
  - Data Types          :  80.0/100 (4 findings)
  - Constant Columns    : 100.0/100 (0 findings)
  - High Cardinality    : 100.0/100 (0 findings)
  - Dataset Structure   :  45.0/100 (5 findings)
  - Leakage Risk        : 100.0/100 (0 findings)


### Key Takeaways
- `fs.load()` normalizes files and DataFrames into a standard schema contract.
- `fs.profile()` executes ultra-fast Polars computations to extract shape and column descriptors.
- `fs.review()` runs 8 automated reviewers to inspect missingness, data types, and target leakage risk.
- `fs.score()` transforms review findings into an explainable 0–100 quality scorecard.